In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV

# =========================
# USTAWIENIA
# =========================

RANDOM_STATE = 42
TARGET_COL = "Is1Winner"

TRAIN_PATH = "total_w.csv"
PRED1_PATH = "predict_w_1.csv"
PRED2_PATH = "predict_w_2.csv"

SUBMISSION_PATH = "submission_w.csv"

best_params_extratrees = {
    "criterion": "log_loss",
    "max_depth": 10,
    "max_features": 0.9,
    "min_samples_leaf": 8,
    "min_samples_split": 8,
    "n_estimators": 421
}

# dla kobiet najlepszy był feature set diff
BEST_FEATURE_SET = "diff"

manual_drop_cols = [
    "Season",
    "1TeamID",
    "2TeamID",
    "TeamID_2",
    "ID"
]

In [2]:
# =========================
# FUNKCJE POMOCNICZE
# =========================

def build_feature_set_single(df, target_col, manual_drop_cols, feature_set_name):
    drop_cols = [c for c in manual_drop_cols if c in df.columns]

    if target_col in df.columns:
        drop_cols.append(target_col)

    base_cols = [c for c in df.columns if c not in drop_cols]

    X_raw = df[base_cols].copy()

    cols_2 = [c for c in base_cols if c.endswith("_2")]
    cols_1 = [c[:-2] for c in cols_2 if c[:-2] in base_cols]

    X_diff = pd.DataFrame(index=df.index)
    for c in cols_1:
        X_diff[f"{c}_diff"] = df[c] - df[f"{c}_2"]

    if feature_set_name == "raw":
        return X_raw
    elif feature_set_name == "diff":
        return X_diff
    elif feature_set_name == "raw_diff":
        return pd.concat([X_raw, X_diff], axis=1)
    else:
        raise ValueError(f"Nieznany feature set: {feature_set_name}")


def align_and_clean_features(X_train, X_pred, missing_threshold=0.60, corr_threshold=0.995):
    X_train = X_train.copy()
    X_pred = X_pred.copy()

    common_cols = [c for c in X_train.columns if c in X_pred.columns]
    X_train = X_train[common_cols].copy()
    X_pred = X_pred[common_cols].copy()

    all_nan_cols = [c for c in X_train.columns if X_train[c].isna().all()]
    high_missing_cols = [c for c in X_train.columns if X_train[c].isna().mean() > missing_threshold]

    nunique = X_train.nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()

    drop_cols = sorted(set(all_nan_cols + high_missing_cols + constant_cols))

    X_train = X_train.drop(columns=drop_cols, errors="ignore")
    X_pred = X_pred.drop(columns=drop_cols, errors="ignore")

    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_pred = pd.DataFrame(
        imputer.transform(X_pred),
        columns=X_pred.columns,
        index=X_pred.index
    )

    if X_train.shape[1] > 1:
        corr = X_train.corr(numeric_only=True).abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        high_corr_cols = [c for c in upper.columns if any(upper[c] > corr_threshold)]
    else:
        high_corr_cols = []

    X_train = X_train.drop(columns=high_corr_cols, errors="ignore")
    X_pred = X_pred.drop(columns=high_corr_cols, errors="ignore")

    return X_train, X_pred

In [3]:
# =========================
# WCZYTANIE DANYCH
# =========================

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
pred1_df = pd.read_csv(PRED1_PATH, low_memory=False)
pred2_df = pd.read_csv(PRED2_PATH, low_memory=False)

pred_df = pd.concat([pred1_df, pred2_df], axis=0, ignore_index=True)

print("train_df:", train_df.shape)
print("pred1_df:", pred1_df.shape)
print("pred2_df:", pred2_df.shape)
print("pred_df :", pred_df.shape)

train_df: (961, 175)
pred1_df: (32851, 175)
pred2_df: (32852, 175)
pred_df : (65703, 175)


In [4]:
# =========================
# KONTROLA KOLUMN
# =========================

if TARGET_COL not in train_df.columns:
    raise ValueError(f"W total_w.csv brakuje targetu: {TARGET_COL}")

if "ID" not in pred_df.columns:
    raise ValueError("W plikach predict_w_1.csv / predict_w_2.csv brakuje kolumny ID")

In [5]:
# =========================
# BUDOWA CECH
# =========================

y_train = train_df[TARGET_COL].astype(int).copy()

X_train = build_feature_set_single(
    train_df,
    target_col=TARGET_COL,
    manual_drop_cols=manual_drop_cols,
    feature_set_name=BEST_FEATURE_SET
)

X_pred = build_feature_set_single(
    pred_df,
    target_col=TARGET_COL,
    manual_drop_cols=manual_drop_cols,
    feature_set_name=BEST_FEATURE_SET
)

X_train, X_pred = align_and_clean_features(X_train, X_pred)

print("X_train:", X_train.shape)
print("X_pred :", X_pred.shape)

X_train: (961, 83)
X_pred : (65703, 83)


In [6]:
# =========================
# TRENING MODELU
# =========================

base_model = ExtraTreesClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_params_extratrees
)

base_model.fit(X_train, y_train)

cal_model = CalibratedClassifierCV(
    estimator=base_model,
    method="sigmoid",
    cv=3
)

cal_model.fit(X_train, y_train)

print("Model wytrenowany.")

Model wytrenowany.


In [7]:
# =========================
# PREDYKCJE
# =========================

pred_proba = cal_model.predict_proba(X_pred)[:, 1]

print("Pred mean:", pred_proba.mean())
print("Pred min :", pred_proba.min())
print("Pred max :", pred_proba.max())

Pred mean: 0.48949513308099135
Pred min : 0.05028741197321054
Pred max : 0.9414789701227043


In [9]:
# =========================
# SUBMISSION: ID, Pred
# =========================

submission = pred_df[["ID"]].copy()
submission["Pred"] = pred_proba.clip(0, 1)

print(submission.head())
print(submission.shape)
print(submission.columns.tolist())

               ID      Pred
0  2026_3101_3102  0.544607
1  2026_3101_3103  0.605484
2  2026_3101_3104  0.066470
3  2026_3101_3105  0.498642
4  2026_3101_3106  0.623313
(65703, 2)
['ID', 'Pred']


In [10]:
# =========================
# ZAPIS
# =========================

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Zapisano plik submission: {SUBMISSION_PATH}")

Zapisano plik submission: submission_w.csv
